In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
#First step is to create a Dataset. Here we have created Dataset locally.
import pandas as pd

#QA
inputs = [
    "Where did the story of mankind begin, and why is that region significant?",
    
    "What major change occurred during the Agricultural Revolution?",
    
    "What were some key contributions of Ancient Greece to human civilization?",
    
    "How did the Industrial Revolution transform societies?"
]

outputs = [
    "The story of mankind began in Africa, often called the 'Cradle of Humanity,' because it is where the earliest human ancestors and Homo sapiens evolved.",

    "The Agricultural Revolution marked the transition from hunting and gathering to cultivating crops and domesticating animals, leading to permanent settlements, population growth, and the rise of civilizations.",

    "Ancient Greece contributed significantly to philosophy, science, mathematics, politics, and the arts, with thinkers such as Socrates, Plato, and Aristotle influencing modern thought and the development of democratic ideas.",

    "The Industrial Revolution transformed societies through innovations such as the steam engine and mechanized manufacturing, leading to rapid urbanization, economic growth, improved transportation, and the foundation of modern technologies."
]
#Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)
#Write to CSV
csv_path = "D:/LLMOps/data/Mankind_qa.csv"
df.to_csv(csv_path, index=False)

In [5]:
#Now we will create Dataset inside LangSmith and upload the CSV file to it.
from langsmith import Client
client = Client()
dataset_name = "Mankind_QA_Dataset"
#Store
dataset = client.create_dataset(
  dataset_name = dataset_name,
  description = "A dataset containing question-answer pairs about the history of mankind.",
)
client.create_examples(
  inputs = [{"question": q} for q in inputs],
  outputs = [{"answer": a} for a in outputs],
  dataset_id = dataset.id
)



{'example_ids': ['68e3882e-6ecf-4134-986d-3be199354959',
  '302e896b-dda5-4432-a0c7-5398940b92f2',
  '11bdbd68-1216-4293-a791-b5d6b451784e',
  'c802bf47-f969-4a7c-91b4-3317a276f5a7'],
 'count': 4,
 'as_of': '2026-06-23T08:22:28.062773852Z'}

In [9]:
#Now we have to run our RAG application here.
import sys
sys.path.append("D:/LLMOps")
from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

#Simple file adapter for local file paths
class LocalFileAdapter:
  """Adapter for local file paths to work with ChatIngestor."""
  def __init__(self, file_path: str):
    self.path = Path(file_path)
    self.name = self.path.name

  def getbuffer(self) -> bytes:
    return self.path.read_bytes()
  
def answer_ai_report_question(
  inputs: dict,
  data_path: str="D:/LLMOps/data/History_Mankind.txt",
  chunk_size: int=1000,
  chunk_overlap: int=200,
  k: int=3
) -> dict:
	"""Answer questions about the history of mankind using a RAG approach."""
	try:
		#Extract question from inputs
		question = inputs.get("question", "")
		if not question:
			return {"answer": "No question provided."}
		#Check if file exists
		if not Path(data_path).exists():
			return {"answer": f"Data file not found at {data_path}."}
		#Create file Adapter
		file_adapter =LocalFileAdapter(data_path)

		#Build index using ChatIngestor
		ingestor = ChatIngestor(
			temp_base = "data",
			faiss_base = "faiss_index",
			use_session_dirs = True
		)
		#build retriever
		ingestor.built_retriever(
			uploaded_files = [file_adapter],
			chunk_size = chunk_size,
			chunk_overlap = chunk_overlap,
			k = k
		)

		#Get the session ID and index path
		session_id = ingestor.session_id
		index_path = f"faiss_index/{session_id}"

		#Create RAG instance and load retriever
		rag = ConversationalRAG(session_id=session_id)
		rag.load_retriever_from_faiss(index_path=index_path, k=k, index_name=os.getenv("FAISS_INDEX_NAME", "index"))

		#Get answer
		answer = rag.invoke(question, chat_history=[])

		return {"answer": answer}

	except Exception as e:
		return {"answer": f"An error occurred: {str(e)}"}




In [10]:
#Test the function with a sample question
test_input = {"question": "Where did the story of mankind begin, and why is that region significant?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("Answer:", result["answer"])

{"timestamp": "2026-06-23T08:24:57.477862Z", "level": "info", "event": "Running in LOCAL mode (.env loaded)"}
{"timestamp": "2026-06-23T08:24:57.478869Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from environment"}
{"timestamp": "2026-06-23T08:24:57.479610Z", "level": "info", "event": "Loaded GROQ_API_KEY from environment"}
{"available_keys": ["GOOGLE_API_KEY", "GROQ_API_KEY"], "timestamp": "2026-06-23T08:24:57.481060Z", "level": "info", "event": "API key manager initialized"}
{"config_keys": ["app", "embedding_model", "retriever", "chunking", "vectorstore", "llm", "logging"], "timestamp": "2026-06-23T08:24:57.502899Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260623135457_3915527a", "temp_dir": "data\\session_20260623135457_3915527a", "faiss_dir": "faiss_index\\session_20260623135457_3915527a", "sessionlized": true, "timestamp": "2026-06-23T08:24:57.507168Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "History_Mankind.tx

Question: Where did the story of mankind begin, and why is that region significant?
Answer: According to the provided context, the story of mankind began in Africa, often referred to as the "Cradle of Humanity." This region is significant because scientists believe that the earliest ancestors of modern humans evolved from primate species millions of years ago.


In [11]:
from langsmith.evaluation import evaluate

#Example: Test with all Mankind_qa questions
for i, q in enumerate(inputs, 1):
	test_input = {"question": q}
	result = answer_ai_report_question(test_input)
	print(f"Q{i}: {q}")
	print(f"A{i}: {result['answer']}\n")
	print("-"*80+"\n")

{"timestamp": "2026-06-23T08:26:29.813623Z", "level": "info", "event": "Running in LOCAL mode (.env loaded)"}
{"timestamp": "2026-06-23T08:26:29.815447Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from environment"}
{"timestamp": "2026-06-23T08:26:29.817225Z", "level": "info", "event": "Loaded GROQ_API_KEY from environment"}
{"available_keys": ["GOOGLE_API_KEY", "GROQ_API_KEY"], "timestamp": "2026-06-23T08:26:29.819485Z", "level": "info", "event": "API key manager initialized"}
{"config_keys": ["app", "embedding_model", "retriever", "chunking", "vectorstore", "llm", "logging"], "timestamp": "2026-06-23T08:26:29.830078Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260623135629_d12a8b5b", "temp_dir": "data\\session_20260623135629_d12a8b5b", "faiss_dir": "faiss_index\\session_20260623135629_d12a8b5b", "sessionlized": true, "timestamp": "2026-06-23T08:26:29.834627Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "History_Mankind.tx

Q1: Where did the story of mankind begin, and why is that region significant?
A1: According to the provided context, the story of mankind began in Africa, often referred to as the "Cradle of Humanity." This region is significant because scientists believe that the earliest ancestors of modern humans evolved from primate species millions of years ago.

--------------------------------------------------------------------------------



HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

Q2: What major change occurred during the Agricultural Revolution?
A2: According to the provided context, one of the most significant transformations in human history that occurred during the Agricultural Revolution was the transition from relying solely on hunting and gathering to cultivating crops and domesticating animals. This marked a major shift in how humans produced food and had a profound impact on population growth, settlement patterns, and societal development.

--------------------------------------------------------------------------------



HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

Q3: What were some key contributions of Ancient Greece to human civilization?
A3: According to the provided context, Ancient Greece made profound contributions to:

* Philosophy
* Science
* Mathematics
* Politics
* Arts

Thinkers such as Socrates, Plato, and Aristotle explored questions about ethics, knowledge, and human nature that continue to influence modern thought. Greek city-states like Athens experimented with forms of democracy.

--------------------------------------------------------------------------------



HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

Q4: How did the Industrial Revolution transform societies?
A4: According to the provided context, the Industrial Revolution transformed societies by:

* Cities expanding rapidly as people moved from rural areas to seek factory work
* Economic growth accelerating
* New social classes emerging
* Technological progress improving living standards for many, though working conditions were often harsh during the early stages of industrialization

--------------------------------------------------------------------------------



In [13]:
from langsmith.evaluation import evaluate
from langsmith.evaluation import LangChainStringEvaluator

#Evaluators
qa_evaluator = LangChainStringEvaluator("cot_qa")
dataset_name = "Mankind_QA_Dataset"

#Run Evaluation using our RAG function and the dataset we created in LangSmith
experiment_results = evaluate(
	answer_ai_report_question,
	data=dataset_name,
	evaluators=qa_evaluator,
	experiment_prefix="test-Mankind-qa-rag",
	#Experiment metadata
	metadata={
		"variant": "RAG with FAISS and Human Kind Report",
		"chunk_size": 1000,
		"chunk_overlap": 200,
		"k": 5
	},
)

ImportError: cannot import name 'LangChainStringEvaluator' from 'langsmith.evaluation' (d:\LLMOps\.venv-1\Lib\site-packages\langsmith\evaluation\__init__.py)

In [ ]:
#Below is the evaluation box (the big box in the diagram)

In [14]:
from langsmith.evaluation import evaluate
from langchain_ollama import ChatOllama

dataset_name = "Mankind_QA_Dataset"

# Judge model used by the evaluator
your_eval_llm = ChatOllama(
    model="llama3:latest",
    temperature=0
)

# LangSmith-compatible evaluator
def qa_evaluator(run, example):
    predicted_answer = (run.outputs or {}).get("answer", "")
    reference_answer = (example.outputs or {}).get("answer", "")
    question = (example.inputs or {}).get("question", "")

    if not predicted_answer:
        return {
            "key": "cot_qa",
            "score": 0,
            "comment": "No answer returned by the application."
        }

    if not reference_answer:
        return {
            "key": "cot_qa",
            "score": 0,
            "comment": "No reference answer found in dataset example."
        }

    prompt = f"""
You are evaluating a question-answering system.

Question:
{question}

Reference answer:
{reference_answer}

Predicted answer:
{predicted_answer}

Decide whether the predicted answer correctly answers the question and is consistent with the reference answer.

Think step by step, then output exactly in this format:
Score: 1
Reason: <short explanation>

or

Score: 0
Reason: <short explanation>
"""

    judgment = your_eval_llm.invoke(prompt).content.strip()
    score = 1 if "Score: 1" in judgment else 0

    return {
        "key": "cot_qa",
        "score": score,
        "comment": judgment
    }

experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=[qa_evaluator],
    experiment_prefix="test-Mankind-qa-rag",
    metadata={
        "variant": "RAG with FAISS and Human Kind Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5
    },
)

View the evaluation results for experiment: 'test-Mankind-qa-rag-6f283132' at:
https://smith.langchain.com/o/50c035b4-d952-4ae5-bfa3-6c6d335cf168/datasets/203321c7-8a48-4414-87dd-2ae965b89e9d/compare?selectedSessions=05cf8e3e-1035-4880-8be1-23bf3984251c




0it [00:00, ?it/s]{"timestamp": "2026-06-23T08:31:49.144436Z", "level": "info", "event": "Running in LOCAL mode (.env loaded)"}
{"timestamp": "2026-06-23T08:31:49.145980Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from environment"}
{"timestamp": "2026-06-23T08:31:49.147559Z", "level": "info", "event": "Loaded GROQ_API_KEY from environment"}
{"available_keys": ["GOOGLE_API_KEY", "GROQ_API_KEY"], "timestamp": "2026-06-23T08:31:49.150493Z", "level": "info", "event": "API key manager initialized"}
{"config_keys": ["app", "embedding_model", "retriever", "chunking", "vectorstore", "llm", "logging"], "timestamp": "2026-06-23T08:31:49.157242Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260623140149_7eb8bf25", "temp_dir": "data\\session_20260623140149_7eb8bf25", "faiss_dir": "faiss_index\\session_20260623140149_7eb8bf25", "sessionlized": true, "timestamp": "2026-06-23T08:31:49.161618Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "

In [13]:
import langsmith
print(langsmith.__version__)

import langsmith.evaluation as ev
print(dir(ev))

0.8.16
['EvaluationResult', 'EvaluationResults', 'RunEvaluator', 'StringEvaluator', 'aevaluate', 'aevaluate_existing', 'evaluate', 'evaluate_comparative', 'evaluate_existing', 'run_evaluator']


In [15]:
"""We can even make a combined multiple Evaluators by just defining them inside the combined_evaluators=[first,second] and then passing it inside the evaluate function."""

'We can even make a combined multiple Evaluators by just defining them inside the combined_evaluators=[first,second] and then passing it inside the evaluate function.'